<a href="https://colab.research.google.com/github/dannys0n/Generative-AI-Works-CS394/blob/main/generate_synthetic_for_tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate Synthetic CS2 Commentary Training Data

Use this notebook to generate synthetic rows for a Counter-Strike 2 commentary dataset with strict structured outputs.

The notebook expects seed examples shaped like your wrapper format:

```json
{
  "input": {
    "context": {
      "score": {"CT": 8, "T": 5},
      "alive_players": [...]
    },
    "previous_events": [...],
    "current_events": [...],
    "request": {
      "mode": "event_bundle" | "idle_color" | "idle_conversation",
      "lines": [
        {
          "caster": "play_by_play" | "color",
          "style": "play_by_play_event" | "play_by_play_follow_up" | "idle_color"
        }
      ]
    }
  }
}
```

It generates synthetic rows shaped like this:

```json
{
  "input": { "...": "..." },
  "output": {
    "lines": [
      {
        "caster": "play_by_play",
        "text": "Target eliminated. Control maintained."
      },
      {
        "caster": "color",
        "text": "Good job. Blue held that perfectly."
      }
    ]
  }
}
```

## Install Dependencies

In [25]:
!pip install -q openai tqdm

## Data generation settings


In [26]:
NUM_TRAIN_EXAMPLES = 5  # @param {type:"number"}
NUM_VAL_EXAMPLES = 1  # @param {type:"number"}
NUM_TEST_EXAMPLES = 1  # @param {type:"number"}

TEMPERATURE = 0.9  # @param {type:"number"}
TOP_P = 0.95  # @param {type:"number"}
FREQUENCY_PENALTY = 0.5  # @param {type:"number"}
PRESENCE_PENALTY = 0.15  # @param {type:"number"}
MAX_ROW_RETRIES = 5  # @param {type:"number"}

DATA_FOLDER = "./.data/generated_cs2"
SEED_EXAMPLES_FILE = "./training_wrapper_pretty.jsonl"  # upload or mount this in Colab
# Keep small for token efficiency
MAX_SEED_EXAMPLES = 50  # @param {type:"number"}
SEED_EXAMPLES_PER_PROMPT = 3  # @param {type:"number"}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5.4-nano"

# Keep this defined so existing code does not break if file loading falls back
INLINE_SEED_EXAMPLES = []

ALLOWED_REQUEST_MODES = [
    "event_bundle",
    "idle_color",
    "idle_conversation",
]


## Seed examples and schema


In [27]:
import json
from pathlib import Path

def load_seed_examples(seed_file: str, inline_examples: list[dict], max_examples: int) -> list[dict]:
    examples: list[dict] = []
    path = Path(seed_file)

    if path.exists():
        text = path.read_text(encoding="utf-8")

        chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
        if chunks:
            for chunk in chunks:
                record = json.loads(chunk)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)
        else:
            for raw_line in text.splitlines():
                line = raw_line.strip()
                if not line:
                    continue
                record = json.loads(line)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)

    for record in inline_examples:
        if isinstance(record, dict) and isinstance(record.get("input"), dict):
            examples.append(record)

    if max_examples > 0:
        examples = examples[:max_examples]

    return examples


SEED_EXAMPLES = load_seed_examples(SEED_EXAMPLES_FILE, INLINE_SEED_EXAMPLES, MAX_SEED_EXAMPLES)
print(f"Loaded {len(SEED_EXAMPLES)} seed examples")
if SEED_EXAMPLES:
    print(json.dumps(SEED_EXAMPLES[0], indent=2, sort_keys=True))


Loaded 27 seed examples
{
  "input": {
    "context": {
      "alive_players": [
        {
          "map_callout": "Double Stack",
          "name": "Dayo",
          "team": "CT"
        },
        {
          "map_callout": "Goose",
          "name": "Ferris",
          "team": "CT"
        },
        {
          "map_callout": "Goose",
          "name": "Harvey",
          "team": "CT"
        },
        {
          "map_callout": "T Spawn",
          "name": "Mayer",
          "team": "CT"
        },
        {
          "map_callout": "T Spawn",
          "name": "Telsen",
          "team": "CT"
        },
        {
          "map_callout": "Close",
          "name": "Walt",
          "team": "CT"
        },
        {
          "map_callout": "Upper Tunnels",
          "name": "Greymouth",
          "team": "T"
        },
        {
          "map_callout": "Mid",
          "name": "Larry",
          "team": "T"
        },
        {
          "map_callout": "Mid",
          "name":

## Model for structured output


In [28]:
from typing import Any, Literal
from pydantic import BaseModel, Field, ConfigDict, model_validator


class StrictBaseModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class RequestLine(StrictBaseModel):
    caster: Literal["play_by_play", "color"]
    style: Literal["play_by_play_event", "play_by_play_follow_up", "idle_color"]


class RequestSpec(StrictBaseModel):
    mode: Literal["event_bundle", "idle_color", "idle_conversation"]
    lines: list[RequestLine] = Field(default_factory=list)


class SyntheticInput(StrictBaseModel):
    context: dict[str, Any]
    previous_events: list[dict[str, Any]] = Field(default_factory=list)
    current_events: list[dict[str, Any]] = Field(default_factory=list)
    request: RequestSpec


class OutputLine(StrictBaseModel):
    caster: Literal["play_by_play", "color"]
    text: str = Field(min_length=1, max_length=240)


class SyntheticOutput(StrictBaseModel):
    lines: list[OutputLine] = Field(min_length=1, max_length=3)


class SyntheticTrainingRow(StrictBaseModel):
    input: SyntheticInput
    output: SyntheticOutput

    @model_validator(mode="after")
    def validate_output_against_request(self):
        request_lines = self.input.request.lines
        output_lines = self.output.lines
        mode = self.input.request.mode

        if not request_lines:
            raise ValueError("input.request.lines must not be empty")

        if mode == "event_bundle":
            if len(output_lines) != len(request_lines):
                raise ValueError(
                    f"event_bundle requires exactly {len(request_lines)} output lines, got {len(output_lines)}"
                )
        else:
            if not (1 <= len(output_lines) <= len(request_lines)):
                raise ValueError(
                    f"{mode} must return between 1 and {len(request_lines)} lines, got {len(output_lines)}"
                )

        for idx, output_line in enumerate(output_lines):
            if output_line.caster != request_lines[idx].caster:
                raise ValueError(
                    f"output line {idx} caster {output_line.caster!r} does not match "
                    f"request caster {request_lines[idx].caster!r}"
                )

        return self


LINES_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "synthetic_output_lines",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "lines": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "caster": {
                                "type": "string",
                                "enum": ["play_by_play", "color"]
                            },
                            "text": {
                                "type": "string"
                            }
                        },
                        "required": ["caster", "text"]
                    },
                    "minItems": 1,
                    "maxItems": 3
                }
            },
            "required": ["lines"]
        }
    }
}

## Get OpenRouter API key


In [29]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

## Synthetic generation functions


In [30]:
import copy
import json
import os
import random
import openai

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)


FEW_SHOT_EXAMPLES = [
    {
        "input": {
            "context": {
                "score": {"CT": 2, "T": 5},
                "alive_players": [
                    {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    {"name": "Maru", "team": "T", "map_callout": "B Site"}
                ]
            },
            "previous_events": [],
            "current_events": [
                {
                    "event_type": "kill",
                    "killer": {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    "victim": {"name": "Maru", "team": "T"}
                }
            ],
            "request": {
                "mode": "event_bundle",
                "lines": [
                    {"caster": "play_by_play", "style": "play_by_play_event"},
                    {"caster": "color", "style": "play_by_play_follow_up"}
                ]
            }
        },
        "output": {
            "lines": [
                {"caster": "play_by_play", "text": "Colin deletes Maru at B Doors."},
                {"caster": "color", "text": "That anchor duel matters. B stays sealed, for now."}
            ]
        }
    },
    {
        "input": {
            "context": {
                "score": {"CT": 11, "T": 10},
                "alive_players": [
                    {"name": "Walt", "team": "CT", "map_callout": "Top Mid"},
                    {"name": "Uri", "team": "T", "map_callout": "Xbox"},
                    {"name": "Mayer", "team": "CT", "map_callout": "Connector"}
                ]
            },
            "previous_events": [
                {"event_type": "grenade_detonated", "grenade_type": "smoke", "detonation_callout": "Xbox"}
            ],
            "current_events": [],
            "request": {
                "mode": "idle_conversation",
                "lines": [
                    {"caster": "play_by_play", "style": "idle_color"},
                    {"caster": "color", "style": "idle_color"},
                    {"caster": "play_by_play", "style": "idle_color"}
                ]
            }
        },
        "output": {
            "lines": [
                {"caster": "play_by_play", "text": "Mid remains contested. No commitment yet."},
                {"caster": "color", "text": "Everybody is waiting for a mistake. That is very healthy and normal."},
                {"caster": "play_by_play", "text": "Connector pressure could decide the round."}
            ]
        }
    },
]


SYSTEM_PROMPT = """
You generate structured commentary lines for a CS2 casting system.

STRICT REQUIREMENTS:
- Return valid JSON only.
- Follow the provided JSON schema exactly.
- Return only one object with a top-level key: lines.
- Do not return input.
- Do not return explanations.
- Keep all lines speakable for TTS.

CASTERS:

1. PLAY-BY-PLAY
- Inspired by the Portal 2 Announcer.
- Clinical, precise, efficient.
- Focus on what just happened or the most important tactical state.
- Short and authoritative.
- Usually 1 to 5 words. Hard cap: 8 words.

2. COLOR
- Inspired by the Portal 2 Turret.
- Quirky, polite, lightly unstable, but insightful.
- Can expand on positioning, map control, utility value, trading, timing, and consequences.
- Can react to the other caster.
- Usually 3 to 12 words. Hard cap: 15 words.

MODE RULES:
- event_bundle: return exactly the requested number of lines.
- idle_color and idle_conversation: return between 1 and the number requested.
- Preserve caster order.
- Each returned line must use the same caster as the corresponding request line.

QUALITY RULES:
- This is Counter-Strike 2.
- the map is Dust2
- Prefer exact player names and callouts when available.
- Ground output in current_events first, then previous_events, then context.
- Avoid generic filler like:
  - huge kill
  - big round
  - what a play
  - finds a frag
  - locks it down
  - steps up
- Avoid repeating the same phrase inside a line.
- Do not break character.
""".strip()


def choose_request() -> dict:
    mode = random.choice(ALLOWED_REQUEST_MODES)

    if mode == "event_bundle":
        return {
            "mode": "event_bundle",
            "lines": [
                {"caster": "play_by_play", "style": "play_by_play_event"},
                {"caster": "color", "style": "play_by_play_follow_up"}
            ],
        }

    if mode == "idle_conversation":
        return {
            "mode": "idle_conversation",
            "lines": [
                {"caster": "play_by_play", "style": "idle_color"},
                {"caster": "color", "style": "idle_color"},
                {"caster": "play_by_play", "style": "idle_color"}
            ],
        }

    return {
        "mode": "idle_color",
        "lines": [
            {"caster": "color", "style": "idle_color"},
            {"caster": "color", "style": "idle_color"},
            {"caster": "color", "style": "idle_color"}
        ],
    }


def build_messages(input_obj: dict) -> list[dict]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for example in FEW_SHOT_EXAMPLES:
        messages.append({
            "role": "user",
            "content": json.dumps(example["input"], ensure_ascii=False)
        })
        messages.append({
            "role": "assistant",
            "content": json.dumps(example["output"], ensure_ascii=False)
        })

    messages.append({
        "role": "user",
        "content": json.dumps(input_obj, ensure_ascii=False)
    })
    return messages


def validate_row_semantics(row: SyntheticTrainingRow) -> None:
    request = row.input.request
    output_lines = row.output.lines

    if request.mode == "event_bundle" and len(output_lines) != len(request.lines):
        raise ValueError("event_bundle must return exactly the requested number of lines")

    if request.mode in {"idle_color", "idle_conversation"}:
        if not (1 <= len(output_lines) <= len(request.lines)):
            raise ValueError(f"{request.mode} must return between 1 and {len(request.lines)} lines")

    for idx, output_line in enumerate(output_lines):
        text = output_line.text.strip()
        if not text:
            raise ValueError(f"output line {idx} is empty")

        if output_line.caster == "play_by_play" and len(text.split()) > 18:
            raise ValueError(f"play_by_play line {idx} is too long: {text}")

        if output_line.caster == "color" and len(text.split()) > 28:
            raise ValueError(f"color line {idx} is too long: {text}")


def pick_input_template(seed_examples: list[dict]) -> dict:
    valid = [ex for ex in seed_examples if isinstance(ex, dict) and "input" in ex]
    if not valid:
        raise ValueError("No seed examples with an 'input' field were found.")
    chosen = copy.deepcopy(random.choice(valid))
    return chosen["input"]


def generate_synthetic_row(seed_examples: list[dict]) -> SyntheticTrainingRow:
    if not seed_examples:
        raise ValueError("No seed examples loaded. Upload or point SEED_EXAMPLES_FILE to your JSONL file.")

    input_obj = pick_input_template(seed_examples)
    input_obj["request"] = choose_request()

    response = client.chat.completions.create(
        model=DATAGEN_MODEL,
        messages=build_messages(input_obj),
        response_format=LINES_RESPONSE_FORMAT,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        frequency_penalty=FREQUENCY_PENALTY,
        presence_penalty=PRESENCE_PENALTY,
    )

    output_obj = json.loads(response.choices[0].message.content)

    row = SyntheticTrainingRow.model_validate({
        "input": input_obj,
        "output": output_obj,
    })

    validate_row_semantics(row)
    return row

## Dataset generation functions


In [31]:
from tqdm import tqdm
from pathlib import Path


def generate_dataset(num_examples: int, filename: str) -> None:
    if "SEED_EXAMPLES" not in globals():
        raise RuntimeError("SEED_EXAMPLES is not defined. Run the seed-loading cell first.")
    if not SEED_EXAMPLES:
        raise RuntimeError("SEED_EXAMPLES is empty. Check SEED_EXAMPLES_FILE and reload seeds.")

    Path(filename).parent.mkdir(parents=True, exist_ok=True)

    with open(filename, "w", encoding="utf-8") as f:
        for idx in tqdm(range(num_examples)):
            row = None
            last_error = None
            for attempt in range(1, MAX_ROW_RETRIES + 1):
                try:
                    row = generate_synthetic_row(SEED_EXAMPLES)
                    break
                except Exception as error:
                    last_error = error
                    print(f"Error generating row {idx} on attempt {attempt}/{MAX_ROW_RETRIES}: {error}")
            if row is None:
                raise RuntimeError(f"Failed to generate row {idx} after {MAX_ROW_RETRIES} attempts") from last_error
            f.write(row.model_dump_json() + "\n")
            f.flush()

## Generate all the data!


In [32]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y-%m-%d-%H:%M:%S')
TRAIN_FILE = f"{DATA_FOLDER}/train_{timestamp}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{timestamp}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{timestamp}.jsonl"

GENERATED_FILES = []

if NUM_TRAIN_EXAMPLES > 0:
    generate_dataset(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
    GENERATED_FILES.append(("train", TRAIN_FILE))

if NUM_VAL_EXAMPLES > 0:
    generate_dataset(NUM_VAL_EXAMPLES, VALID_FILE)
    GENERATED_FILES.append(("valid", VALID_FILE))

if NUM_TEST_EXAMPLES > 0:
    generate_dataset(NUM_TEST_EXAMPLES, TEST_FILE)
    GENERATED_FILES.append(("test", TEST_FILE))

for split_name, split_path in GENERATED_FILES:
    print(f"{split_name}: {split_path}")


100%|██████████| 1/1 [00:00<00:00,  1.22it/s]

train: ./.data/generated_cs2/train_2026-04-15-00:39:32.jsonl
valid: ./.data/generated_cs2/valid_2026-04-15-00:39:32.jsonl
test: ./.data/generated_cs2/test_2026-04-15-00:39:32.jsonl


In [33]:
# Preview generated rows
import json
from pathlib import Path

for split_name, split_path in GENERATED_FILES:
    preview_path = Path(split_path)
    if not preview_path.exists():
        continue

    print(f"=== {split_name} preview ===")
    for raw_line in preview_path.read_text(encoding="utf-8").splitlines()[:3]:
        parsed = json.loads(raw_line)
        print(json.dumps(parsed, indent=2, ensure_ascii=False))
        print("---")


=== train preview ===
{
  "input": {
    "context": {
      "alive_players": [
        {
          "map_callout": "Long Doors",
          "name": "Dayo",
          "team": "CT"
        },
        {
          "map_callout": "Cross",
          "name": "Ferris",
          "team": "CT"
        },
        {
          "map_callout": "Long Doors",
          "name": "Harvey",
          "team": "CT"
        },
        {
          "map_callout": "Goose",
          "name": "Telsen",
          "team": "CT"
        },
        {
          "map_callout": "Ramp",
          "name": "Walt",
          "team": "CT"
        },
        {
          "map_callout": "Blue",
          "name": "Larry",
          "team": "T"
        }
      ],
      "score": {
        "CT": 7,
        "T": 5
      }
    },
    "previous_events": [
      {
        "detonation_callout": null,
        "event_type": "grenade_detonated",
        "grenade_type": "inferno",
        "owner_player": {}
      }
    ],
    "current_events": 